# Module 5: Perception — CNN
### BWSI SGAI 2026

By the end of this module you'll know:
- What the CNN does and how the game uses its output
- How to swap `CNNClassifier` for `OracleClassifier` to diagnose problems
- Where to extend triage value logic for humanoids

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## How Does the Game See?

In Module 4 you traced a full game turn: `base_env` calls `classifier.classify(humanoid)` and gets back an array of probabilities. The game never looks at the image directly — it only sees those numbers.

This module covers the **perception layer**: the code that turns a humanoid image into a probability distribution the rest of the game can act on.

```
image file
   ↓  classify(humanoid)
[zombie_prob, healthy_prob, injured_prob, corpse_prob]
   ↓
 game logic  /  RL recommender  /  UI display
```

## Where does this live in the code?

```
bwsi-sgai/
├── gameplay/
├── models/                     ← this module
│   ├── HumanoidClassifier.py   ← now
│   ├── CNNClassifier.py        ← now
│   ├── OracleClassifier.py     ← now
│   ├── ActionRecommender.py
│   ├── RLRecommender.py
│   ├── HeuristicRecommender.py
│   ├── DefaultCNN.py
│   ├── DQNAgent.py
│   ├── DQN.py
│   └── weights/
├── game_interfaces/
│   └── data_parser.py          ← also this module
├── ui/
├── training/
└── tests/
```

*The three perception files are `HumanoidClassifier.py`, `CNNClassifier.py`, and `OracleClassifier.py`. We also cover `data_parser.py` — it controls how humanoids are built from the CSV.*

*`bwsi-sgai/models/HumanoidClassifier.py`*

## HumanoidClassifier — the Contract

```
bwsi-sgai/
├── models/
│   ├── HumanoidClassifier.py   ← here
│   ├── CNNClassifier.py
│   └── OracleClassifier.py
```

Open `models/HumanoidClassifier.py`. It's short — about 100 lines including comments.

This file defines an **Abstract Base Class (ABC)**. An ABC sets up a contract: any class that inherits from it *must* implement every method marked `@abstractmethod`, or Python raises a `TypeError` when you try to instantiate it.

Python docs: [Abstract Base Classes](https://docs.python.org/3/library/abc.html)

*`bwsi-sgai/models/HumanoidClassifier.py`*

## HumanoidClassifier — Orient

```python
from abc import ABC, abstractmethod
import numpy as np
from gameplay.enums import State
from gameplay.humanoid import Humanoid

class HumanoidClassifier(ABC):
    """Decides what a humanoid IS."""

    @abstractmethod
    def classify(self, humanoid: Humanoid) -> np.ndarray:
        """Return a probability distribution over all humanoid classes.

        Returns:
            1-D float32 array of shape (4,) in class order:
            [zombie, healthy, injured, corpse]
        """
```

One abstract method. That's the entire contract.

**Why use an ABC here?** Any class that implements `classify()` can be swapped in without changing the game loop. `CNNClassifier`, `OracleClassifier`, and any classifier *you* write all satisfy the same contract — the game doesn't care which one it's talking to.

*`bwsi-sgai/models/HumanoidClassifier.py`*

## `probs_to_class()` — module-level helper

Also in `HumanoidClassifier.py`, above the class definition:

```python
def probs_to_class(probs: np.ndarray) -> tuple[State, float]:
    """Convert a probability array to a (label, confidence) pair.

    Returns:
        (label, confidence) where label is a State enum value
        and confidence is a float in [0.0, 1.0].
    """
    idx = int(np.argmax(probs))
    # COUPLING: list(State) order must match [zombie, healthy, injured, corpse]
    return list(State)[idx], float(probs[idx])
```

**Use `probs_to_class(probs)` — not `np.argmax(probs)` directly.**  
`np.argmax` returns an `int`. `probs_to_class` returns a `State` enum value and a confidence float — which is what the game loop and UI actually need.

*`bwsi-sgai/models/HumanoidClassifier.py`*

## `probs_to_class()` — Touch

Run the cell below. You're calling `probs_to_class` with a hand-crafted probability array — no classifier or image needed.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
from models.HumanoidClassifier import probs_to_class
import numpy as np

# class order: [zombie, healthy, injured, corpse]
probs = np.array([0.7, 0.1, 0.1, 0.1], dtype=np.float32)

label, conf = probs_to_class(probs)
print(f"probs  = {probs}")
print(f"label  = {label}")
print(f"conf   = {conf:.2f}")
print()

# What if the model is uncertain?
uncertain = np.array([0.3, 0.3, 0.3, 0.1], dtype=np.float32)
label2, conf2 = probs_to_class(uncertain)
print(f"uncertain probs = {uncertain}")
print(f"label  = {label2}")
print(f"conf   = {conf2:.2f}  (low confidence!)")

probs  = [0.7 0.1 0.1 0.1]
label  = State.ZOMBIE
conf   = 0.70

uncertain probs = [0.3 0.3 0.3 0.1]
label  = State.ZOMBIE
conf   = 0.30  (low confidence!)


INSTRUCTOR NOTES:
Ask: what does a confidence of 0.3 tell you about the CNN's certainty?
Point out that the game displays this confidence value in the UI when a classifier is active.
Ask: why return a State enum value instead of just the string 'zombie'? (Type safety, IDE completion, guards against typos.)

## Now: the CNN implementation

```
bwsi-sgai/
├── models/
│   ├── HumanoidClassifier.py
│   ├── CNNClassifier.py        ← now
│   ├── OracleClassifier.py
│   └── weights/
│       └── baselineCNN.pth       ← loaded at startup
```

*`CNNClassifier` is where the trained model lives. It reads an image file and returns 4 probabilities.*

*`bwsi-sgai/models/CNNClassifier.py`*

## CNNClassifier — Locate

Open `models/CNNClassifier.py`. It's about 130 lines.

`CNNClassifier` inherits from `HumanoidClassifier` (the ABC) and implements `classify()`. It:

1. **Loads trained weights** from `models/weights/baselineCNN.pth` at construction time
2. **Preprocesses** the humanoid's image (resize, normalize)
3. **Runs a forward pass** through `DefaultCNN` (the neural network architecture)
4. **Returns softmax probabilities** as a `(4,)` NumPy array

If the weights file is missing, you get a `FileNotFoundError` immediately — fail fast, clear error, rather than silent bad output.

*`bwsi-sgai/models/CNNClassifier.py`*

## CNNClassifier — Orient: constructor

```python
class CNNClassifier(HumanoidClassifier):

    def __init__(
        self,
        num_classes: int = 4,
        model_file: str = CLASSIFIER_MODEL_PATH,   # 'models/weights/baselineCNN.pth'
        img_data_root: str = "data",
    ) -> None:
        if not os.path.exists(model_file):
            raise FileNotFoundError(
                f"CNN weights not found at '{model_file}'. "
                "Train the model first ('python main.py train --type cnn') or "
                "update CLASSIFIER_MODEL_PATH in gameplay/constants.py."
            )
        # ... sets up device, image_transforms, loads model ...
```

Two things to notice:
- `model_file` defaults to the path defined in `gameplay/constants.py` — one place to change it
- The `FileNotFoundError` is raised at construction time, not later — you find out immediately if weights are missing

*`bwsi-sgai/models/CNNClassifier.py`*

## CNNClassifier — Orient: `classify()`

```python
    def classify(self, humanoid: Humanoid) -> np.ndarray:
        with Image.open(os.path.join(self.img_data_root, humanoid.fp)) as image_:
            tensor = (
                self.image_transforms(image_.convert("RGB"))
                .float()
                .unsqueeze(0)      # add batch dimension: (3,512,512) -> (1,3,512,512)
                .to(self.device)
            )
        with torch.no_grad():
            outputs = self.net(tensor)
            return torch.nn.functional.softmax(outputs, 1)[0].cpu().numpy()
        # Returns: float32 array shape (4,)
        # class order: [zombie, healthy, injured, corpse]
```

The output order `[zombie, healthy, injured, corpse]` is a **hard coupling**:  
it must match the output layer of `DefaultCNN`, the `State` enum order in `enums.py`, and `probs_to_class()`.  
**Never reorder the `State` enum entries** — it silently breaks classification everywhere.

> **The interesting seam:** why does `classify()` take a `Humanoid` object instead of just an image path?  
> Because `Humanoid` carries both `fp` (the image path) and `state` (the ground-truth label).  
> `CNNClassifier.classify()` only uses `humanoid.fp`. `OracleClassifier.classify()` only uses `humanoid.state`.  
> Both satisfy the same interface — the caller doesn't know (or care) which one it has.

*`bwsi-sgai/models/CNNClassifier.py`*

## CNNClassifier — Touch

First, check whether the weights file exists on your machine.  
Run the cell below — if weights are missing, skip ahead to `OracleClassifier` for the exercise.

In [3]:
import os

weights_path = "models/weights/baselineCNN.pth"
print("Weights exist:", os.path.exists(weights_path))

if not os.path.exists(weights_path):
    print()
    print("No weights file found.")
    print("Options:")
    print("  1. Train: python main.py train --type cnn")
    print("  2. Use OracleClassifier instead (no weights needed) -- see next slide")

Weights exist: True


INSTRUCTOR NOTES:
If the weights file is missing for most students, pivot to OracleClassifier for the live demo.
The exercise at the end of this module uses OracleClassifier as the diagnostic tool, so no weights are needed for the exercise itself.
If most students have weights, optionally run the CNNClassifier touch cell below. Skip it if hardware is slow.

In [4]:
import os

# Only run this cell if the weights file exists (check the cell above first)
weights_path = "models/weights/baselineCNN.pth"

if os.path.exists(weights_path):
    from models.CNNClassifier import CNNClassifier
    from models.HumanoidClassifier import probs_to_class
    import pandas as pd

    classifier = CNNClassifier(img_data_root="data")
    print("CNNClassifier loaded successfully")
    print(f"Device: {classifier.device}")
    print()

    # Classify one humanoid from the CSV
    from game_interfaces.data_parser import DataParser
    parser = DataParser(data_dir="data", seed=42)
    humanoid = parser.get_random()
    probs = classifier.classify(humanoid)
    label, conf = probs_to_class(probs)

    print(f"Humanoid ground truth: {humanoid.state}")
    print(f"CNN probabilities:     {probs.round(3)}")
    print(f"CNN prediction:        {label}  (confidence {conf:.2f})")
    print(f"Correct?               {humanoid.state == label.value}")
else:
    print("Skipping -- weights not found. See OracleClassifier on the next slides.")

CNNClassifier loaded successfully
Device: cuda

Humanoid ground truth: healthy
CNN probabilities:     [0.    0.999 0.    0.   ]
CNN prediction:        State.HEALTHY  (confidence 1.00)
Correct?               True


## Now: the Oracle

```
bwsi-sgai/
├── models/
│   ├── HumanoidClassifier.py
│   ├── CNNClassifier.py
│   └── OracleClassifier.py     ← now
```

*`OracleClassifier` requires no weights and is always correct. Use it to isolate perception problems from RL problems.*

*`bwsi-sgai/models/OracleClassifier.py`*

## OracleClassifier — Orient

```python
class OracleClassifier(HumanoidClassifier):
    """Perfect-knowledge classifier that reads humanoid.state directly.

    humanoid.state is set from the CSV ground-truth label at parse time,
    so this bypasses all image inference entirely.
    """

    def classify(self, humanoid: Humanoid) -> np.ndarray:
        idx = MAP_CLASS_STR_TO_INT[humanoid.state]
        one_hot = np.zeros(len(MAP_CLASS_INT_TO_STR), dtype=np.float32)
        one_hot[idx] = 1.0
        return one_hot
        # e.g. zombie -> [1., 0., 0., 0.]
        # e.g. healthy -> [0., 1., 0., 0.]
```

No weights. No `__init__` arguments. Always returns a perfect one-hot distribution.

Same interface as `CNNClassifier` — the game loop can't tell the difference.

*`bwsi-sgai/models/OracleClassifier.py`*

## OracleClassifier — Diagnostic Tool

**Why does a "cheat mode" classifier exist?**

It's a diagnostic instrument. When something goes wrong in the game, you need to know *where* the problem is:

| Situation | What it tells you |
|-----------|------------------|
| Game plays **well** with Oracle, **poorly** with CNN | The CNN is the bottleneck. Retrain or improve it. |
| Game plays **poorly** with Oracle | The RL policy needs work — perception isn't the issue. |
| Weights file is missing | Use Oracle as a fallback until you retrain. |

Swap in anywhere `CNNClassifier` is used in `main.py`.

*`bwsi-sgai/models/OracleClassifier.py`*

## OracleClassifier — Touch

Run the cell below. No weights needed.

In [5]:
from models.OracleClassifier import OracleClassifier
from models.HumanoidClassifier import probs_to_class
from game_interfaces.data_parser import DataParser

print("Initialized, no weights needed")

# Classify a few humanoids and verify the oracle is always correct
oc = OracleClassifier()
parser = DataParser(data_dir="data", seed=42)

print()
print(f"{'Ground truth':<15} {'Oracle prediction':<20} {'Confidence':<10} {'Correct?'}")
print("-" * 60)
for _ in range(5):
    h = parser.get_random()
    probs = oc.classify(h)
    label, conf = probs_to_class(probs)
    correct = h.state == label.value
    print(f"{h.state:<15} {str(label):<20} {conf:<10.2f} {correct}")

Initialized, no weights needed

Ground truth    Oracle prediction    Confidence Correct?
------------------------------------------------------------
healthy         State.HEALTHY        1.00       True
injured         State.INJURED        1.00       True
healthy         State.HEALTHY        1.00       True
healthy         State.HEALTHY        1.00       True
healthy         State.HEALTHY        1.00       True


INSTRUCTOR NOTES:
Ask: why is confidence always 1.0 for OracleClassifier? (It returns a one-hot vector, so np.argmax always gives probability 1.0.)
Ask: when would you use OracleClassifier in a real research project? (Ablation study -- isolate one component.)
Point out that 'Correct?' is always True -- that's the point of the oracle.

## Now: how humanoids get their values

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
│   └── data_parser.py          ← now
├── ui/
├── training/
└── tests/
```

*`data_parser.py` contains `DataParser`, `datarow_to_state()`, and `datarow_to_value()`.  
It's where each humanoid's triage priority is assigned before the game loop ever starts.*

*`bwsi-sgai/game_interfaces/data_parser.py`*

## data_parser.py — Locate

Open `game_interfaces/data_parser.py`. It has three things:

```
data_parser.py
├── class DataParser
│   ├── __init__(data_dir, seed)  -- loads CSV, sets up unvisited pool
│   ├── get_random()             -- draws one humanoid without replacement
│   └── reset()                  -- refills the pool for a new episode
├── def datarow_to_state(data_row)  -- maps CSV row -> state string
└── def datarow_to_value(data_row)  -- maps CSV row -> float in [0.0, 1.0]
```

The two module-level functions are marked `# can be customized` in the source.  
`datarow_to_value()` is the most common extension point.

*`bwsi-sgai/game_interfaces/data_parser.py`*

## `datarow_to_value()` — Orient

```python
# can be customized
def datarow_to_value(data_row: pd.Series, rng: random.Random | None = None) -> float:
    """Assign a numeric priority value to a humanoid from its metadata row.

    Returns a float in [0.0, 1.0] representing triage priority.
    Higher values = higher rescue priority.

    Available columns: Class, Injured, Gender, Item, Filename.
    """
    # Default: uniform random -- every humanoid gets an independent random value.
    # To customize: inspect data_row columns and return a computed float.
    _rng = rng if rng is not None else random
    return _rng.random()
```

By default, every humanoid gets a **random** priority. The `rng` parameter is the seeded `random.Random` instance from `DataParser` — so with `DataParser(seed=42)`, values are reproducible.

*`bwsi-sgai/game_interfaces/data_parser.py`*

## `datarow_to_value()` — The Seam

This function is the most accessible place to change how the game evaluates humanoids.

**Example extensions:**

```python
def datarow_to_value(data_row, rng=None):
    # Prioritize humanoids carrying a bow (rare item)
    if data_row["Item"] == "Bow":
        return 0.9
    return 0.3

def datarow_to_value(data_row, rng=None):
    # Injured humanoids are highest priority
    if data_row["Injured"] is True:
        return 1.0
    return 0.5
```

**One constraint:** the return value **must stay in `[0.0, 1.0]`**.  
Values outside this range silently corrupt the reward calculation.

Available columns from the CSV: `Class`, `Injured`, `Gender`, `Item`, `Filename`.

*`bwsi-sgai/game_interfaces/data_parser.py`*

## `datarow_to_value()` — Touch

`datarow_to_value()` takes a `pd.Series` from the metadata CSV. It can't be called in isolation — load the dataframe first.

In [6]:
import pandas as pd
from game_interfaces.data_parser import datarow_to_value

df = pd.read_csv("data/consolidated_metadata.csv")
row = df.iloc[0]

print("Columns in the metadata CSV:")
print(list(df.columns))
print()
print("First row:")
print(row.to_dict())
print()

# Call datarow_to_value with default rng (non-reproducible)
value = datarow_to_value(row)
print(f"datarow_to_value(row) = {value:.4f}")
print(f"(random by default -- run again to see it change)")

Columns in the metadata CSV:
['Class', 'Injured', 'Gender', 'Item', 'Filename']

First row:
{'Class': 'Default', 'Injured': False, 'Gender': 'Female', 'Item': 'Bow', 'Filename': 'consolidated_dataset/test_00000.png'}

datarow_to_value(row) = 0.5295
(random by default -- run again to see it change)


*`bwsi-sgai/game_interfaces/data_parser.py`*

## `datarow_to_state()` — Brief mention

`datarow_to_value()` isn't the only extension point. There's also:

```python
def datarow_to_state(data_row: pd.Series) -> str:
    img_class  = str(data_row["Class"]).strip()
    img_injured = data_row["Injured"]

    # Extension point: uncomment and use these columns
    # img_gender = data_row['Gender']
    # img_item   = data_row['Item']

    if img_class == "Default":
        state = State.HEALTHY.value
        if injured:
            state = State.INJURED.value
    else:
        state = State.ZOMBIE.value
        if injured:
            state = State.CORPSE.value
    return state
```

This is where `Class` + `Injured` columns map to `State` values.  
Edit this to change what makes a humanoid a zombie or a corpse.

INSTRUCTOR NOTES:
Ask: what columns are available for customizing value? (Class, Injured, Gender, Item, Filename)
Ask: if a student wants injured humanoids to always have value 1.0, which function do they edit? (datarow_to_value)
Ask: if a student wants female humanoids to be treated as healthy regardless of injury, which function do they edit? (datarow_to_state -- and they should understand the ethics of that choice)
Point to feedback_lecture_format.md: kaitiakitanga context applies if students ask about demographic-based triage logic.

## Exercise: Swap CNNClassifier → OracleClassifier

**Time: under 5 minutes. No weights needed.**

You're going to replace the CNN with the Oracle in `main.py`, run a game, and observe what changes.

**Why?** If the game plays *better* with the Oracle than the CNN, the CNN is the bottleneck.  
This is how you distinguish a perception problem from a policy problem.

---

**Before you start:** confirm the Oracle is importable.

In [7]:
from models.OracleClassifier import OracleClassifier
print("OracleClassifier ready -- no weights needed")

OracleClassifier ready -- no weights needed


## Exercise: Steps 1–3 — Edit `main.py`

Open `main.py` in VSCode.

**Step 1.** Near the top of the file, find the existing classifier import:
```python
from models.CNNClassifier import CNNClassifier
```
Add this line directly below it:
```python
from models.OracleClassifier import OracleClassifier
```

**Step 2.** Search for `CNNClassifier(img_data_root=data_dir)` in `main.py` (Ctrl+F).  
It appears in two places: `_make_components()` and `_run_auto()`.  
Replace **both** with:
```python
OracleClassifier()
```

**Step 3.** Save the file.

## Exercise: Step 4 — Run and Compare

Run this command in the terminal:

```bash
python main.py -m auto --log --seed 42
```

**Before you run, predict:**
- Will the score be higher or lower than with the CNN?
- Will the reward change a lot, or a little?
- Why?

After it finishes, check `summary.csv` for the `reward` column.

```bash
python -c "import pandas as pd; print(pd.read_csv('summary.csv').tail(3).to_string())"
```

## Exercise: Step 5 — Interpret and Revert

**Interpreting the result:**

| What you observe | What it means |
|-----------------|---------------|
| Oracle score **much higher** than CNN | The CNN misclassifies enough humanoids to matter. Retrain or improve it. |
| Oracle score **similar** to CNN | The CNN is already doing well. The RL policy is the binding constraint. |
| Oracle score **still low** | The RL policy itself needs work — even perfect perception doesn't help. |

**Step 5. Revert both edits in `main.py`:**
- Remove the `OracleClassifier` import
- Change `OracleClassifier()` back to `CNNClassifier(img_data_root=data_dir)` in both places

Or use `git diff` to review what changed, and `git restore main.py` to reset.

INSTRUCTOR NOTES:
Walk through the edit locations in main.py before students start -- show Ctrl+F for CNNClassifier.
After the run, pull up summary.csv together and compare reward values.
Key teaching moment: the oracle gives the RL agent perfect information. If the policy still underperforms with perfect information, the policy is the problem.
Students who finish early: try running with --seed 0 and --seed 1 with both classifiers. Does the oracle always beat the CNN?

## Pop Quiz — Navigation

These are navigation questions, not recall. Which file do you open, and why?

---

**Q1.** You want to write a classifier that always predicts `'healthy'` — no matter what the image looks like.  
Which method must your new class implement, and where is its contract defined?

A) `recommend()` in `models/ActionRecommender.py`  
B) `classify()` in `models/HumanoidClassifier.py`  
C) `get_random()` in `game_interfaces/data_parser.py`  
D) `get_score()` in `gameplay/scorekeeper.py`

**Answer: B — `classify()` in `models/HumanoidClassifier.py`**  
Inherit from `HumanoidClassifier` and implement `classify(humanoid) -> np.ndarray`.  
Return `np.array([0., 1., 0., 0.], dtype=np.float32)` — 100% probability on healthy (index 1).  
Swap it in wherever `CNNClassifier` is instantiated in `main.py`.

## Pop Quiz — Navigation

**Q2.** You want to make triage value depend on whether the humanoid carries a Bow.  
Which function do you edit, and in which file?

A) `classify()` in `models/CNNClassifier.py`  
B) `get_cumulative_reward()` in `gameplay/scorekeeper.py`  
C) `datarow_to_value()` in `game_interfaces/data_parser.py`  
D) `get_random()` in `game_interfaces/data_parser.py`

**Answer: C — `datarow_to_value()` in `game_interfaces/data_parser.py`**  
Check `data_row["Item"] == "Bow"` and return a high value (e.g. `0.9`).  
The return must stay in `[0.0, 1.0]` — values outside that range silently corrupt the reward.  
`datarow_to_state()` (same file) maps columns to state — different extension, different purpose.

## Pop Quiz — Navigation

**Q3.** The game plays well with `OracleClassifier` but poorly with `CNNClassifier`. What's the bottleneck, and what do you do about it?

A) The RL policy — retrain with more episodes  
B) The CNN — retrain or improve the perception model  
C) The triage value function — edit `datarow_to_value()`  
D) The reward weights — adjust `HEADCOUNT_WEIGHT` in `constants.py`

**Answer: B — the CNN is the bottleneck**  
The RL policy is fine — it plays well when it gets perfect perception from the Oracle.  
The problem is that `CNNClassifier` is misclassifying enough humanoids to hurt the score.  
Fix: retrain with `python main.py train --type cnn`, or improve the training data / augmentation in `training/train_cnn.py`.

## What You Now Know

- `models/HumanoidClassifier.py` — the ABC; **implement `classify()` to plug in your own classifier**
- `models/CNNClassifier.py` — the trained CNN; requires `models/weights/baselineCNN.pth`
- `models/OracleClassifier.py` — ground-truth oracle; **use it to diagnose perception vs. policy problems**
- `game_interfaces/data_parser.py` — `datarow_to_value()` is **the extension point for triage logic**

The key diagnostic pattern:
```
Oracle good, CNN bad  →  retrain the CNN
Oracle bad            →  fix the RL policy
```

---

**Next: Module 6 — Decision (RL)**  
The RL agent receives those probability arrays and picks an action. We'll open `models/RLRecommender.py` and `models/HeuristicRecommender.py` — and edit the heuristic rules directly.